## Create and test the functions to load in the relevant data

In [1]:
import numpy as np
import pandas as pd
import datetime

import matplotlib.pyplot as plt
import seaborn as sns
# import plotly as ply

from macro_etf.data_loader import load_yfinance_data, load_fred_data
from macro_etf.common import PROCESSED_DATA_DIR


### FRED data

Data is loaded using the fredapi Python package, which requires an API and downloads data from [fred.stlouisfed.org](https://fred.stlouisfed.org/docs/api/api_key.html).

These data correspond to broad market health indicators which I will use together to measure overall market health.

A description for each pulled feature follows:

**cpi**: Consumer Price Index. A measure of inflation that tracks how the basic cost of goods and services changes over time. This works by tracking a *basket* of goods (e.g., food, gas, housing, healthcare, etc) over time and determining the cost of such a basket. There are multiple ways that baskets are defined, and thus multiple forms of CPI.

**unemployment**: The percentage of people who actively want to work but currently aren't, typically due to a job shortage or low wages.

**yield_spread**: The difference in the interest rate between a 10 year US treasury bond and a 2 year bond. Typically the yield of a 10 year bond is higher than that of a 2 year bond, since money is being committed for a longer term. If the economy starts to falter and investors lose confidence in its near-term health then they will sell their short term bonds (2 year) and buy long term bonds (10 year) which causes the price of short term bonds to decrease and long term bonds to increase. During times of extreme economic chaos the yield_spread can become negative, meaning the effective interest of the 2 year bond is higher than the 10 year bond. This yield spread inversion has preceeded every US recession since 1970, making it a useful predictor of market health. Note that there hasn't been a recession every single time this inversion has occured.

**fed_funds_rate**: The interest rate at which banks lend money to each other overnight. The Fed increases this to reduce inflation when the economy is too hot. If this value is increasing over time then the Fed is actively trying to slow the economy down. If this goes up then it becomes more expensive to borrow money and so becomes more difficult for companies to grow. Also, any business that has large loans will have to start making larger interest payments which means they will have less capital for growth.

**credit_spread**: The difference in yield between a corporate bond and a US treasury bond. A US bond is considered a very safe investment as it isn't expected to default on its debt. A corporate bond on the other hand is more risky and so investors expect a higher rate of return. If the economy is doing well then investors do not expect corporations to go bust and so are more willing to consider interest rates slightly higher than the US treasury bonds. If the economy is not doing well then investors will demand higher interest rates, which causes the credit_spread to grow. The credit_spread often reacts faster to market conditions than stocks do.

**financial_stress**: This is a composite index that attempts to track overall market stress. This works by taking 18 stress-based indices, performing principal component analysis (PCA) and using the first component as the index. PCA identifies base vectors that describe the greatest amount of variability in a dataset. By taking the first principal component, we are in essence creating a new index that combines the 18 other indices weighted by importance.

**ted_spread (DISCONTINUED)**: the difference between the 3-month LIBOR rate (what banks charge to borrow from each other) and the 3-month US treasury bond yield. This reflects how much the banks no longer trust each other to remain solvent overnight. If this suddenly jumps then banks think there is a high probability that other banks will fail, which is a terrible sign for the economy. It also means that banks are no longer lending to businesses and customers which again negatively influences the economy.

**industrial_production**: a direct measure of how much physical stuff the US economy is generating (mining, manufacturing, utilities). The value is relative to the 2017 value, with values over 100 representing higher output. This is a useful index in that it directly measures physical output and is not dependent on how people are *feeling* about the economy.

**retail_sales**: measures the total dollar value of sales made through US retailers each month. This value makes up ~70% of the US GDP and so is a good measure of how the US economy is doing right now. Note that this is **not** inflation adjustested, meaning that I should adjust it myself to get an accurate picture of overall economic health. Especially important right now as inflation has been relatively high recently.

**leading_indicators (DISCONTINUED)**: this is another composite index built that attempts to predict how well the economy will be doing in 6-12 months from now. It incorporates building permits, jobless claims, stock prices, yield spread, consumer expectations, and others. This may be redundant as I already have many of these indices in my data, but it may serve as a useful comparison index in terms of how well my future model performs.

**real_gdp**: Captures all spending in a country: consumer spending, investments, government spending, net exports. Nominal GDP measures the GDP using the current currency value, while the real GDP uses one reference value for the currency by adjusting for inflation. This is only released quarterly rather than monthly, so I will have to figure out how to deal with that issue in my data/model.

**consumer_sentiment**: A sort of vibe-check on how the average consumer feels about the economy. The value is determined by surveying 500 homes and asking questions such as "Are you better or worse off financially than a year ago?" or "Is now a good time to buy a major household item (car, hot tub)?" If the resulting measure is above 100 then consumers are optimistic, below then pessimistic. This can capture uncertainty in the market due to global events that have not yet had a chance to effect the markets (e.g., Iran war oil crisis).

**housing_starts**: the number of houses or housing units that had started construction in the last month. Housing construction is heavily dependent on interest rates because builders require loans to build the homes and buyers need to get mortgages to pay for them. The number of housing starts generally reacts faster to changes in the Fed rate than other measures. This is because building a house requires months of planning and if the builders lose confidence that there will be demand or that it will be too expensive, they will decrease the number of houses they are building.

**building_permits**: Similar to housing starts, but this number should change even earlier. Building permits are approved near the beginning of a construction project and if there is a dropoff then builders are likely pulling back on the number of houses they are going to build. This is a leading indicator. Likely higher co-linear with housing_starts, may be a good idea to just use one, or to consider the ratio between them as a widening gap could indicate an abrupt housing market slowdown.

**m2**: a measure of total money supply in the economy. How much money exists and is readily available for spending. M1 = total physical cash and money in checkings accounts. M2 = M1 + savings accounts + money market funds + CDs / GICs. If this value increases then there is more money in the economy and so inflation rises. If it decreases then there are deflationary pressures to consider. M2 often increases 1-2 years *before* inflation starts to increase, making it a leading indicator. 

**initial_claims**: measures the number of people filing for unemployment claims for the first time each week. This is measured more often than monthly jobs reports, gives a better immediate measure of economic health. 

**job_opennings**: how much employers want to hire people, rather than the number of people without jobs. Employeers often pull back on hiring before they resort to laying people off and so this is a leading indicator of how employers are feeling about their economic future.

In [2]:
df_fred = load_fred_data()
df_fred.describe()

,cpi,unemployment,yield_spread,fed_funds_rate,credit_spread,financial_stress,industrial_production,retail_sales,real_gdp,consumer_sentiment,credit_card_delinquency,personal_savings_rate,housing_starts,building_permits,m2,initial_claims,job_openings
count,376.000000,376.000000,7874.000000,377.000000,7867.000000,1642.000000,377.000000,377.000000,125.000000,376.000000,125.000000,376.000000,377.000000,377.000000,377.000000,1.641000e+03,305.000000
mean,222.767109,5.513032,0.940130,2.566684,2.350629,0.006246,94.639965,360486.872679,17383.091504,84.471277,3.588720,5.747872,1341.355438,1390.217507,10688.709549,3.614022e+05,5492.095082
std,47.771599,1.805625,0.925327,2.230310,0.736542,1.015203,7.952808,128431.286690,3440.200406,14.861401,1.186457,2.907736,396.127190,405.086516,6016.226771,3.158927e+05,2276.417975
min,150.500000,3.400000,-1.080000,0.050000,1.220000,-1.130400,71.188200,174560.000000,11319.951000,49.800000,1.530000,1.400000,478.000000,513.000000,3489.900000,1.900000e+05,2232.000000
25%,181.425000,4.300000,0.210000,0.180000,1.770000,-0.553925,90.652700,259050.000000,14537.580000,73.125000,2.520000,4.500000,1103.000000,1166.000000,5758.100000,2.560000e+05,3768.000000
50%,218.946500,5.000000,0.650000,1.910000,2.230000,-0.214650,97.666300,330301.000000,16920.632000,87.900000,3.670000,5.600000,1380.000000,1435.000000,8718.700000,3.210000e+05,4732.000000
75%,251.067000,5.925000,1.730000,5.070000,2.780000,0.263450,100.863000,428932.000000,20044.077000,95.825000,4.580000,6.400000,1600.000000,1656.000000,14174.600000,3.800000e+05,6966.000000
max,333.979000,14.800000,2.910000,6.540000,6.160000,9.673100,104.100400,662752.000000,24152.656000,112.000000,6.770000,31.800000,2273.000000,2263.000000,23052.300000,6.137000e+06,12301.000000


In [3]:
df_fred.head()

,cpi,unemployment,yield_spread,fed_funds_rate,credit_spread,financial_stress,industrial_production,retail_sales,real_gdp,consumer_sentiment,credit_card_delinquency,personal_savings_rate,housing_starts,building_permits,m2,initial_claims,job_openings
1995-01-01,150.5,5.6,NaN,5.53,NaN,NaN,71.2734,177136.0,11319.951,97.6,3.48,7.3,1407.0,1282.0,3492.4,NaN,NaN
1995-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-03,NaN,NaN,0.15,NaN,1.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,NaN,NaN,0.20,NaN,1.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,NaN,NaN,0.22,NaN,1.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Sync feature sample rates

Currently, the features appear to be sampled at different rates (e.g., monthly v.s. weekly). I will down/up sample some of these as necessary so that they their sample dates align and can be compared.

First, let's determine how often each is sampled:

In [4]:
# Let's calculate the difference (delta) in sampling dates to determine how often each is sampled.
# Specifically, let's note the min, max and median of the deltas as that can tell us something 
# about the delta distribution. If min max and median are all the same, then the index is measured
# consistently. Otherwise it may be affected by holidays and such.
sample_data_delta_df = pd.DataFrame(index = df_fred.columns, columns = ['min', 'max', 'median', 'sample_rate'] )
for col in df_fred.columns:
    sample_dates = df_fred[col].dropna().index
    sample_date_deltas = [(sample_dates[i] - sample_dates[i-1]).days for i in range(1,len(sample_dates))]
    sample_data_delta_df.loc[col, 'min'] = np.min(sample_date_deltas)
    sample_data_delta_df.loc[col, 'max'] = np.max(sample_date_deltas)
    sample_data_delta_df.loc[col, 'median'] = np.median(sample_date_deltas)
    
    #If sample date delta median is between 28 and 31, then sampling rate is monthly.
    if sample_data_delta_df.loc[col, 'median'] >= 28 and sample_data_delta_df.loc[col, 'median'] <= 31:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'monthly'
    #If sample date delta min and max are 1, then sampling rate is daily with no exceptions.
    elif sample_data_delta_df.loc[col, 'min'] == 1 and sample_data_delta_df.loc[col, 'max']==1:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'daily'
    #If sample date delta median is 1 and max > 1, then I will say that sampling is done every weekday. 
    #This may not be strictly true but okay for this project.
    elif sample_data_delta_df.loc[col, 'median'] == 1 and sample_data_delta_df.loc[col, 'max']>1:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'weekdaily'
    #If sample date delta median is 7, then sampling rate is weekly.
    elif sample_data_delta_df.loc[col, 'median'] == 7:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'weekly'
    #If sample date delta median is over 90, then sampling rate is quarterly.
    elif sample_data_delta_df.loc[col, 'median'] >90:
        sample_data_delta_df.loc[col, 'sample_rate'] = 'quarterly'
    

sample_data_delta_df


,min,max,median,sample_rate
cpi,28,61,31.0,monthly
unemployment,28,61,31.0,monthly
yield_spread,1,4,1.0,weekdaily
fed_funds_rate,28,31,31.0,monthly
credit_spread,1,5,1.0,weekdaily
financial_stress,7,7,7.0,weekly
industrial_production,28,31,31.0,monthly
retail_sales,28,31,31.0,monthly
real_gdp,90,92,91.5,quarterly
consumer_sentiment,28,31,31.0,monthly


For CPI and unemployement, why is the maximum delta two months? That implies a gap in the data which I will need to address.

In [5]:
has_month_skip = ['cpi', 'unemployment']
for col in has_month_skip:
    sample_dates = df_fred[col].dropna().index
    sample_date_deltas = [(sample_dates[i] - sample_dates[i-1]).days for i in range(1,len(sample_dates))]
    where_2month_skip = np.arange(len(sample_date_deltas))[np.isclose(sample_date_deltas,61)][0]
    print(sample_dates[(where_2month_skip-2):(where_2month_skip+3)])

DatetimeIndex(['2025-07-01', '2025-08-01', '2025-09-01', '2025-11-01',
               '2025-12-01'],
              dtype='datetime64[us]', freq=None)
DatetimeIndex(['2025-07-01', '2025-08-01', '2025-09-01', '2025-11-01',
               '2025-12-01'],
              dtype='datetime64[us]', freq=None)


From this we can see that the gap in monthly cpi and unemployment records occured in Oct 2025.
A quick google search shows that this occured because of a prolonged government shutdown, resulting in the first ever gap in these records. I will correct for this by filling in a value for the Oct 2025 date using linear inference (take the mean of the value before and after Oct 2025).

In [6]:
previous_month = datetime.datetime.fromisoformat("2025-09-01")
skipped_month = datetime.datetime.fromisoformat("2025-10-01")
next_month = datetime.datetime.fromisoformat("2025-11-01")
df_fred.loc[skipped_month, 'cpi'] = np.mean([df_fred.loc[[previous_month, next_month], 'cpi']]) 
df_fred.loc[skipped_month, 'unemployment'] = np.mean([df_fred.loc[[previous_month, next_month], 'unemployment']]) 

print(df_fred.loc[[previous_month, skipped_month, next_month], 'unemployment'])
print(df_fred.loc[[previous_month, skipped_month, next_month], 'cpi'])

2025-09-01    4.40
2025-10-01    4.45
2025-11-01    4.50
Name: unemployment, dtype: float64
2025-09-01    324.245
2025-10-01    324.654
2025-11-01    325.063
Name: cpi, dtype: float64


Okay, some are sampled daily, some weekly, some monthly and two quarterly. Let's align them.

First, let's decide on a rate to align to. I will choose monthly arbitrarily here, as that provides enough time for these measures to meaningfully move between sample dates, while not being so long as to make future predictions meaningless.

In [7]:
# Let's do the sampling using Pandas.Series.resample() method, which makes all of this nice and easy.

# Resample to month end
sync_to = 'ME' 

resampled_features = {}
# Downsampling from daily to monthly
    # These spreads are generally less spiked and so it should be okay to take the 
    # last value of the month, to capture current conditions at that date.
resampled_features['yield_spread'] =            df_fred['yield_spread'].resample(sync_to).last()
resampled_features['credit_spread'] =           df_fred['credit_spread'].resample(sync_to).last()
# resampled_features['ted_spread'] =              df_fred['ted_spread'].resample(sync_to).last()

# Downsampling from weekly to monthly
    # Take the mean here, as these are more variable and taking the mean allows us 
    # to capture any peaks occurring in the middle of the month.
resampled_features['financial_stress'] =        df_fred['financial_stress'].resample(sync_to).mean()
resampled_features['initial_claims'] =          df_fred['initial_claims'].resample(sync_to).mean()

# Correcting indices for features already sampled monthly
resampled_features['cpi'] =                     df_fred['cpi'].resample(sync_to).last()
resampled_features['unemployment'] =            df_fred['unemployment'].resample(sync_to).last()
resampled_features['fed_funds_rate'] =          df_fred['fed_funds_rate'].resample(sync_to).last()
resampled_features['industrial_production'] =   df_fred['industrial_production'].resample(sync_to).last()
resampled_features['retail_sales'] =            df_fred['retail_sales'].resample(sync_to).last()
# resampled_features['leading_indicators'] =      df_fred['leading_indicators'].resample(sync_to).last()
resampled_features['consumer_sentiment'] =      df_fred['consumer_sentiment'].resample(sync_to).last()
resampled_features['personal_savings_rate'] =   df_fred['personal_savings_rate'].resample(sync_to).last()
resampled_features['housing_starts'] =          df_fred['housing_starts'].resample(sync_to).last()
resampled_features['building_permits'] =        df_fred['building_permits'].resample(sync_to).last()
resampled_features['m2'] =                      df_fred['m2'].resample(sync_to).last()
resampled_features['job_openings'] =            df_fred['job_openings'].resample(sync_to).last()

# Upsampling from quarterly to monthly.
    #ffill() copies the last valid entry into rows with missing values.
resampled_features['real_gdp'] =                df_fred['real_gdp'].resample(sync_to).first().ffill()
resampled_features['credit_card_delinquency'] = df_fred['credit_card_delinquency'].resample(sync_to).first().ffill()


In [8]:
resampled_features

{'yield_spread': 1995-01-31    0.34
 1995-02-28    0.43
 1995-03-31    0.40
 1995-04-30    0.47
 1995-05-31    0.41
               ... 
 2026-02-28    0.59
 2026-03-31    0.51
 2026-04-30    0.52
 2026-05-31    0.47
 2026-06-30    0.34
 Freq: ME, Name: yield_spread, Length: 378, dtype: float64,
 'credit_spread': 1995-01-31    1.33
 1995-02-28    1.53
 1995-03-31    1.49
 1995-04-30    1.51
 1995-05-31    1.66
               ... 
 2026-02-28    1.80
 2026-03-31    1.79
 2026-04-30    1.70
 2026-05-31    1.57
 2026-06-30    1.51
 Freq: ME, Name: credit_spread, Length: 378, dtype: float64,
 'financial_stress': 1995-01-31   -0.247475
 1995-02-28   -0.314225
 1995-03-31   -0.267120
 1995-04-30   -0.314475
 1995-05-31   -0.286300
                 ...   
 2026-02-28   -0.547500
 2026-03-31   -0.315275
 2026-04-30   -0.577175
 2026-05-31   -0.721880
 2026-06-30   -0.864300
 Freq: ME, Name: financial_stress, Length: 378, dtype: float64,
 'initial_claims': 1995-01-31    333500.0
 1995-02-28    3

In [9]:
#Shift some values to account for report delays.
#Basically, these features are dated by the time that they describe, rather
#than what is known about these feaures at that time. So, in order to build
#a model that predicts future market conditions based on what is currently
#known, we will need to shift some of these features so that the date
#corresponds what is known at that time.

release_lag_months = {
    'unemployment': 1,
    'industrial_production': 1,
    'retail_sales': 1,
    'consumer_sentiment': 1,
    'personal_savings_rate': 1,
    'housing_starts': 1,
    'building_permits': 1,
    'cpi': 1,
    'real_gdp': 1,
    'credit_card_delinquency': 1, 
    'job_openings': 2
}

for feat, lag in release_lag_months.items():
    resampled_features[feat] = resampled_features[feat].shift(periods=lag)

In [18]:
new_fred_df = pd.DataFrame(resampled_features).dropna()
new_fred_df.head(5)

,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,real_gdp,credit_card_delinquency
2001-02-28,0.51,2.88,0.435475,371250.0,175.6,4.2,5.49,91.9020,247339.0,94.7,4.5,1600.0,1699.0,5017.0,5088.0,14183.120,4.81
2001-03-31,0.75,3.04,0.673380,387200.0,176.0,4.2,5.31,91.3034,247289.0,90.6,4.6,1625.0,1656.0,5074.8,5234.0,14183.120,4.81
2001-04-30,1.05,2.73,0.799750,396750.0,176.1,4.3,4.80,91.1162,244514.0,91.5,4.9,1590.0,1659.0,5139.1,5097.0,14183.120,4.81
2001-05-31,1.21,2.65,0.386200,394500.0,176.4,4.4,4.21,90.7891,249113.0,88.4,4.8,1649.0,1666.0,5137.2,4762.0,14271.694,4.94
2001-06-30,1.17,2.65,0.418500,397200.0,177.3,4.3,3.97,90.3555,250250.0,92.0,4.3,1605.0,1665.0,5180.1,4615.0,14271.694,4.94


## Load ticker information

For now, I will focus on 4 tickers: 

**spy**: This is an ETF that aims to follow the S&P 500 exactly. The S&P 500 index is determined by 500 of the largest US companies, weighted by the market-captialization of each company. This means that the index is very top-heavy, with the 10 ten companies (mostly tech darlings) accounting for roughly 35% of the index's value.

**qqq**: This is an ETF that aims to follow the NASDAQ-100 exactly. Similarly to the S&P 500, the Nasdaq-100 is calculated using 100 of the largest US companies, weighted by their market caps. Key differences include NASDAQ not including the financial sector, the companies must be listed on the NASDAQ exchange, there is no profitability requirement (the S&P 500 has one), and there are limits on the proportion each company can take up of the index. The Nasdaq is significantly more concentrated than the S&P 500, with 40-50% of it being taken up by the top 5-6 companies (generally tech darlings).

**oil**: This is a calculated as the average price of a barrel of oil on the futures market. In this case, the futures market is made up of contracts which specify that someone will buy or sell a specific amount of oil at a given date.

**vix**: VIX is a measure of fear and anxiety in the market. If VIX goes up, there are signals that traders believe the market will deteriorate in the next several months. VIX is calculated using the futures market. If buyers think that prices will fall then they will buy more *puts* to protect their portfolios. If many of these are being purchased then their price will increase. VIX measures that increase and uses it to gauge market sentiment.

In [11]:
ticker_dfs = load_yfinance_data()

Let's merge the ticker data together and take a look at what information is available

In [12]:
ticker_df = pd.concat(ticker_dfs).unstack(level=0)
#Fix column names
ticker_df.columns = ticker_df.columns.to_series().map('_'.join)
ticker_df.head()

,Open_spy,Open_qqq,Open_vix,Open_oil,High_spy,High_qqq,High_vix,High_oil,Low_spy,Low_qqq,...,Dividends_vix,Dividends_oil,Stock Splits_spy,Stock Splits_qqq,Stock Splits_vix,Stock Splits_oil,Capital Gains_spy,Capital Gains_qqq,Capital Gains_vix,Capital Gains_oil
Date,,,,,,,,,,,,,,,,,,,,,
1995-01-03,26.398718,NaN,14.09,NaN,26.479945,NaN,14.71,NaN,26.389693,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1995-01-04,26.561144,NaN,13.87,NaN,26.570169,NaN,14.15,NaN,26.425766,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1995-01-05,26.588220,NaN,13.70,NaN,26.633346,NaN,14.19,NaN,26.543094,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1995-01-06,26.624341,NaN,13.67,NaN,26.714593,NaN,13.76,NaN,26.516038,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN
1995-01-09,26.588226,NaN,13.53,NaN,26.624327,NaN,14.08,NaN,26.570176,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN


We can see that each ticker has data for:
- Open
- High
- Low
- Close
- Volume
- Dividends
- Stock Splits
- Capital gains

The first four of these are useful by themselves, though I will likely only make use of Close to track the value of each ticker at the end of the month.

Volume may be useful for tracking overall movement in the market, except for vix which as a measure is not traded.

Dividends I will ignore. These are the dividends that would be paid to you if you held the corresponding weighted mix of company stocks tracked by the index. By paying these dividends out instead of adding the value to the ETFs, the SPY and QQQ follow their respective indices more faithfully.

Stock splits may affect SPY and QQQ by causing their prices to jump up or down depending on the split. I will quickly check to see if there are any and then adjust the prices if necessary.

Capital gains, I will check to see if any are reported, though I suspect not as tickers are not for individual companies.

In [13]:
# Check if there are any dividends, stock splits or capital gains reported

for info_type in ['Stock Splits', 'Dividends', 'Capital Gains']:
    # print(info_type)
    for name, df in ticker_dfs.items():
        if info_type not in df.columns:
            continue
        if np.any(df[info_type]!=0):
            print('Ticker {} contains non-zero entries in column {}'.format(name, info_type))

Ticker qqq contains non-zero entries in column Stock Splits
Ticker spy contains non-zero entries in column Dividends
Ticker qqq contains non-zero entries in column Dividends


Okay, so only SPY and QQQ contain dividends, which I am going to ignore, and so I will drop all three of these from the dataframe.

I am also going to drop open, high and low from the dataset and just use close to track each ticker.

In [14]:
columms_to_drop = [col for col in ticker_df.columns if 'Dividends' in col or
                                                        'Stock Splits' in col or 
                                                        'Capital Gains' in col or
                                                        'Open' in col or
                                                        'High' in col or
                                                        'Low' in col]
columms_to_drop.append("Volume_vix")

ticker_df = ticker_df.drop(columms_to_drop, axis='columns')
ticker_df.head()


,Close_spy,Close_qqq,Close_vix,Close_oil,Volume_spy,Volume_qqq,Volume_oil
Date,,,,,,,
1995-01-03,26.443844,NaN,14.25,NaN,324300.0,NaN,NaN
1995-01-04,26.570169,NaN,13.53,NaN,351800.0,NaN,NaN
1995-01-05,26.570169,NaN,13.50,NaN,89800.0,NaN,NaN
1995-01-06,26.597265,NaN,13.13,NaN,448400.0,NaN,NaN
1995-01-09,26.624327,NaN,13.33,NaN,36800.0,NaN,NaN


I will now need to downsample these as well, so that they can be merged with the FRED data, which is currently sampled monthly.

In [15]:
ticker_df = ticker_df.resample("ME").last()

### Final touch up and then merge the datasets

With both the FRED data and yfinance data loaded and downsampled, I can merge them to create our final dataset for all future analysis.

Let's also move GDP to be the last column as I will use that as a measure of economic health and the last column is easily accessed / found

In [ ]:
final_df = pd.concat([new_fred_df, ticker_df], axis=1, sort=True).dropna()

#Remove any capital letters if I missed any
final_df.columns = final_df.columns.to_series().str.lower().to_list()

#Move real_gdp to the last column as I will use that as an overall measure of economic health
# There is some controversy in this choice, but I will stick with it for now.
cols = final_df.columns.drop("real_gdp").insert(len(final_df.columns)-1,"real_gdp")
final_df = final_df.loc[:,cols]

final_df.to_csv(PROCESSED_DATA_DIR / 'processed_market_data.csv')